In [0]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import(
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

from mlflow.models import infer_signature

catalog_name = spark.sql("SELECT current_catalog()").first()[0]

In [0]:
schema_name = spark.sql("SELECT current_schema()").first()[0]

TRAIN_TABLE = f"{catalog_name}.{schema_name}.fraud_transaction_train"
TEST_TABLE = f"{catalog_name}.{schema_name}.fraud_transaction_test"

train_pdf = spark.table(TRAIN_TABLE).toPandas()
test_pdf = spark.table(TEST_TABLE).toPandas()

X_train = train_pdf.drop(["Class"], axis=1)
y_train = train_pdf["Class"].astype(int)

X_test = test_pdf.drop(["Class"], axis=1)
y_test = test_pdf["Class"].astype(int)

X_test = X_test[X_train.columns]

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)
print("Training fraud cases", int(y_train.sum()))
print("Testing fraud cases", int(y_test.sum()))
      

Training shape: (227845, 30)
Testing shape: (56962, 30)
Training fraud cases 394
Testing fraud cases 98


In [0]:
preprocessor = ColumnTransformer(
    transformers=[
       ("scale", StandardScaler(), ["Time","Amount"])
    ],
    remainder="passthrough"
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            class_weight="balanced",
            random_state=42,
            max_iter=1000
        
        ))
    ]
)

model.fit(X_train, y_train)

print("Model trained")

Model trained


In [0]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

metrics = {
    "precision": precision_score(y_test, y_pred),
    "recall": recall_score(y_test, y_pred),
    "f1": f1_score(y_test, y_pred),
    "roc_auc": roc_auc_score(y_test, y_prob),
    "average_precision": average_precision_score(y_test, y_prob)
}

for metric_name, metric_value in metrics.items():
    print(f"{metric_name}: {metric_value:.4f}")

print("\n Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

print("\n Classification report:")
print(classification_report(y_test, y_pred, zero_division=0))


precision: 0.0610
recall: 0.9184
f1: 0.1144
roc_auc: 0.9722
average_precision: 0.7159

 Confusion matrix:
[[55478  1386]
 [    8    90]]

 Classification report:
              precision    recall  f1-score   support

           0       1.00      0.98      0.99     56864
           1       0.06      0.92      0.11        98

    accuracy                           0.98     56962
   macro avg       0.53      0.95      0.55     56962
weighted avg       1.00      0.98      0.99     56962



In [0]:
mlflow.sklearn.autolog(disable=True)

with mlflow.start_run(run_name="fraud_logistic_regression_baseline") as run:
    mlflow.log_param("model", "Logistic Regression")
    mlflow.log_param("class_weight", "balance")
    mlflow.log_param("max_iter", 1000)
    mlflow.log_param("random_state", 42)
    mlflow.log_param("split_random_state", 42)
    mlflow.log_param("feature_count", X_train.shape[1])
    mlflow.log_param("training_rows", len(X_train))
    mlflow.log_param("testing_rows", len(X_test))

    mlflow.log_metrics(metrics)

    signature = infer_signature(X_train.head(), model.predict(X_train.head()))

    mlflow.sklearn.log_model(
        sk_model=model,
        name = "fraud_model",
        signature=signature,
        input_example=X_train.head(5)
    )

    run_id = run.info.run_id

print("MLflow run ID", run_id)
print("Logged metrics", metrics)

🔗 View Logged Model at: https://dbc-6ec01f7e-de30.cloud.databricks.com/ml/experiments/3215376294506232/models/m-0dfcb3f8b9cb4b819dc93bd7143cf3ca?o=7474654944931794


MLflow run ID b3ca314766bc4afbb27d9bb58180dfbf
Logged metrics {'precision': 0.06097560975609756, 'recall': 0.9183673469387755, 'f1': 0.11435832274459974, 'roc_auc': 0.9721687370080279, 'average_precision': 0.7159122424484009}


In [0]:
client = mlflow.tracking.MlflowClient()
run_info = client.get_run(run_id)

print("Run ID:", run_info.info.run_id)
print("Run Status:", run_info.info.status)

print("Logged paramters:")
print(run_info.data.params)
print("\nLogged metrics:")
print(run_info.data.metrics)
print("\nModel artifact URI:")
print(run_info.info.artifact_uri)

Run ID: b3ca314766bc4afbb27d9bb58180dfbf
Run Status: FINISHED
Logged paramters:
{'class_weight': 'balance', 'feature_count': '30', 'max_iter': '1000', 'model': 'Logistic Regression', 'random_state': '42', 'split_random_state': '42', 'testing_rows': '56962', 'training_rows': '227845'}

Logged metrics:
{'average_precision': 0.7159122424484009, 'f1': 0.11435832274459974, 'precision': 0.06097560975609756, 'recall': 0.9183673469387755, 'roc_auc': 0.9721687370080279}

Model artifact URI:
dbfs:/databricks/mlflow-tracking/3215376294506232/b3ca314766bc4afbb27d9bb58180dfbf/artifacts


In [0]:
import pandas as pd
from datetime import datetime, timezone

catalog_name = spark.sql("SELECT current_catalog()").first()[0]
schema_name = spark.sql("SELECT current_schema()").first()[0]
RUN_REGISTRY_TABLE = (
   f"{catalog_name}.{schema_name}.fraud_model_run_registry"
)
run_registry_pdf = pd.DataFrame([{
   "run_id": run_info.info.run_id,
   "model_name": "fraud_logistic_regression",
   "registered_at_utc": datetime.now(timezone.utc).isoformat()
}])
(spark.createDataFrame(run_registry_pdf)
   .write
   .format("delta")
   .mode("overwrite")
   .saveAsTable(RUN_REGISTRY_TABLE))
print("Saved model run to:", RUN_REGISTRY_TABLE)
print("Run ID:", run_info.info.run_id)

Saved model run to: workspace.default.fraud_model_run_registry
Run ID: b3ca314766bc4afbb27d9bb58180dfbf
